In [8]:
import pandas as pd 
import numpy as np

In [11]:
transactions = pd.read_csv("../data/silver/clean_transactions.csv")
outlets = pd.read_csv("../data/silver/clean_outlet_master.csv")
locations = pd.read_csv("../data/silver/clean_outletCoordinates.csv")
seasonality = pd.read_csv("../data/silver/clean_seasonality.csv")
holidays = pd.read_csv("../data/silver/clean_holidays.csv")

In [18]:
df = transactions.merge(
    outlets,
    left_on='gitOutlet_ID',
    right_on='Outlet_ID',
    how='left'
)

In [21]:
df = df.merge(locations, on='Outlet_ID', how='left')

In [22]:
df = df.merge(seasonality, on=['Distributor_ID', 'Month'], how='left')

In [23]:
df['date'] = pd.to_datetime(df['date'])

KeyError: 'date'

In [24]:
print(df.columns.tolist())

['gitOutlet_ID', 'Year_x', 'Month', 'Distributor_ID', 'SKU_ID', 'Volume_Liters', 'Total_Bill_Value', 'Outlet_ID', 'Outlet_Size', 'Cooler_Count', 'Outlet_Type', 'Latitude', 'Longitude', 'Unnamed: 0', 'Year_y', 'Seasonality_Index']


In [26]:
df['date'] = pd.to_datetime(
    df['Year_x'].astype(str) + '-' +
    df['Month'].astype(str) + '-01'
)

In [28]:
df['quarter'] = df['date'].dt.quarter
df['month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)

In [30]:
df = df.sort_values(['gitOutlet_ID', 'date'])

df['lag_1'] = (
    df.groupby('gitOutlet_ID')['Volume_Liters']
    .shift(1)
)

In [31]:
df['rolling_mean_3'] = (
    df.groupby('gitOutlet_ID')['Volume_Liters']
    .shift(1)
    .rolling(3)
    .mean()
)

In [32]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek
df['weekofyear'] = df['date'].dt.isocalendar().week
df['quarter'] = df['date'].dt.quarter
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

In [37]:
print(holidays.columns.tolist())

['Date', 'Holiday_Name', 'Holiday_Type']


In [42]:
holiday_dates = pd.to_datetime(holidays['Date'])

In [46]:
holidays['Date'] = pd.to_datetime(holidays['Date'])
holiday_months = holidays['Date'].dt.month.unique()

In [47]:
df['is_holiday_month'] = df['month'].isin(holiday_months).astype(int)

In [49]:
print(df.columns.tolist())

['gitOutlet_ID', 'Year_x', 'Month', 'Distributor_ID', 'SKU_ID', 'Volume_Liters', 'Total_Bill_Value', 'Outlet_ID', 'Outlet_Size', 'Cooler_Count', 'Outlet_Type', 'Latitude', 'Longitude', 'Unnamed: 0', 'Year_y', 'Seasonality_Index', 'date', 'quarter', 'month_sin', 'month_cos', 'lag_1', 'rolling_mean_3', 'year', 'month', 'day', 'dayofweek', 'weekofyear', 'is_weekend', 'is_holiday_month']


In [51]:
coords = df[['Latitude', 'Longitude']].dropna()

kmeans = KMeans(n_clusters=10, random_state=42)

df.loc[coords.index, 'geo_cluster'] = kmeans.fit_predict(coords)

In [58]:
#Outlet Average Sales
df['outlet_avg_volume'] = (
    df.groupby('gitOutlet_ID')['Volume_Liters']
    .transform('mean')
)

In [61]:
df['product_avg_volume'] = (
    df.groupby('SKU_ID')['Volume_Liters']
    .transform('mean')
)

In [63]:
y = df['Volume_Liters']

In [62]:
X = df.drop([
    'Volume_Liters',
    'Total_Bill_Value',
    'date'
], axis=1)

In [64]:
X = X.fillna(0)

In [65]:
train = df[df['year'] < 2025]
valid = df[df['year'] == 2025]

X_train = train.drop(['Volume_Liters'], axis=1)
y_train = train['Volume_Liters']

X_valid = valid.drop(['Volume_Liters'], axis=1)
y_valid = valid['Volume_Liters']

In [74]:
X_train.dtypes


gitOutlet_ID                  object
Year_x                         int64
Month                          int64
Distributor_ID                object
SKU_ID                        object
Total_Bill_Value             float64
Outlet_ID_x                   object
Outlet_Size                   object
Cooler_Count                   int64
Outlet_Type                   object
Latitude_x                   float64
Longitude_x                  float64
Unnamed: 0                     int64
Year_y                         int64
Seasonality_Index             object
date                  datetime64[ns]
quarter                        int32
month_sin                    float64
month_cos                    float64
lag_1                        float64
rolling_mean_3               float64
year                           int32
month                          int32
day                            int32
dayofweek                      int32
weekofyear                    UInt32
is_weekend                     int64
i

In [75]:
df = df.drop(columns=[
    'Latitude_y',
    'Longitude_y'
])

In [76]:
df['Seasonality_Index'] = pd.to_numeric(df['Seasonality_Index'], errors='coerce')

In [77]:
cat_features = [
    'gitOutlet_ID',
    'Distributor_ID',
    'SKU_ID',
    'Outlet_ID_x',
    'Outlet_Size',
    'Outlet_Type'
]

In [78]:
X_train.select_dtypes(include='object').columns

Index(['gitOutlet_ID', 'Distributor_ID', 'SKU_ID', 'Outlet_ID_x',
       'Outlet_Size', 'Outlet_Type', 'Seasonality_Index', 'Outlet_ID_y'],
      dtype='object')

In [80]:
X_train.select_dtypes(include='object').columns

Index(['gitOutlet_ID', 'Distributor_ID', 'SKU_ID', 'Outlet_ID_x',
       'Outlet_Size', 'Outlet_Type', 'Seasonality_Index', 'Outlet_ID_y'],
      dtype='object')

In [81]:
df = df.drop(columns=['Outlet_ID_y'])
df = df.rename(columns={'Outlet_ID_x': 'Outlet_ID'})

In [82]:
df['Seasonality_Index'].unique()

array([nan])

In [83]:
df['Seasonality_Index'].isnull().sum()


np.int64(3119877)

In [84]:
df = df.drop(columns=['Seasonality_Index'])

In [86]:
X = df.drop(['Volume_Liters', 'Total_Bill_Value'], axis=1)
y = df['Volume_Liters']

In [87]:
X.dtypes

gitOutlet_ID                  object
Year_x                         int64
Month                          int64
Distributor_ID                object
SKU_ID                        object
Outlet_ID                     object
Outlet_Size                   object
Cooler_Count                   int64
Outlet_Type                   object
Latitude_x                   float64
Longitude_x                  float64
Unnamed: 0                     int64
Year_y                         int64
date                  datetime64[ns]
quarter                        int32
month_sin                    float64
month_cos                    float64
lag_1                        float64
rolling_mean_3               float64
year                           int32
month                          int32
day                            int32
dayofweek                      int32
weekofyear                    UInt32
is_weekend                     int64
is_holiday_month               int64
geo_cluster                  float64
o

In [88]:
X = df.drop([
    'Volume_Liters',
    'Total_Bill_Value',
    'date',
    'Latitude_x',
    'Longitude_x',
    'Unnamed: 0',
    'Year_y'
], axis=1)

In [89]:
cat_features = X.select_dtypes(include='object').columns

In [90]:
train = df[df['year'] < df['year'].max()]
valid = df[df['year'] == df['year'].max()]

X_train = train[X.columns]
y_train = train['Volume_Liters']

X_valid = valid[X.columns]
y_valid = valid['Volume_Liters']

In [92]:
cat_features = list(cat_features)

In [93]:
X_train = X_train.copy()
X_valid = X_valid.copy()

X_valid = X_valid[X_train.columns]

In [95]:
from catboost import CatBoostRegressor

cat_features = list(X_train.select_dtypes(include='object').columns)

model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    verbose=100,
    task_type='CPU',
    thread_count=-1
)
model.fit(
    X_train,
    y_train,
    eval_set=(X_valid, y_valid),
    cat_features=cat_features
)

0:	learn: 90.9486149	test: 90.8587451	best: 90.8587451 (0)	total: 741ms	remaining: 6m 9s
100:	learn: 14.0694797	test: 14.0953520	best: 14.0953520 (100)	total: 1m 48s	remaining: 7m 10s
200:	learn: 11.2032227	test: 11.2715612	best: 11.2715612 (200)	total: 2m 59s	remaining: 4m 26s
300:	learn: 10.1655985	test: 10.2725717	best: 10.2725717 (300)	total: 4m 5s	remaining: 2m 42s
400:	learn: 9.6253699	test: 9.7365378	best: 9.7365378 (400)	total: 5m 11s	remaining: 1m 16s
499:	learn: 9.3306859	test: 9.4593588	best: 9.4593588 (499)	total: 6m 19s	remaining: 0us

bestTest = 9.459358762
bestIteration = 499



CatBoostRegressor(depth=6, iterations=500, learning_rate=0.05, loss_function='RMSE', task_type='CPU', verbose=100)

In [96]:
df['Volume_Liters'].describe()

count    3.119877e+06
mean     5.303629e+01
std      9.537075e+01
min      0.000000e+00
25%      1.031440e+01
50%      2.333684e+01
75%      5.471741e+01
max      1.233045e+03
Name: Volume_Liters, dtype: float64

In [97]:
df['lag_7'] = df.groupby('gitOutlet_ID')['Volume_Liters'].shift(7)
df['lag_14'] = df.groupby('gitOutlet_ID')['Volume_Liters'].shift(14)

In [98]:
df['rolling_mean_7'] = df.groupby('gitOutlet_ID')['Volume_Liters'].transform(lambda x: x.rolling(7).mean())
df['rolling_mean_14'] = df.groupby('gitOutlet_ID')['Volume_Liters'].transform(lambda x: x.rolling(14).mean())

In [99]:
df['trend_1'] = df['lag_1'] - df['rolling_mean_3']

In [100]:
from catboost import CatBoostRegressor

cat_features = list(X_train.select_dtypes(include='object').columns)

model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    verbose=100,
    task_type='CPU',
    thread_count=-1
)
model.fit(
    X_train,
    y_train,
    eval_set=(X_valid, y_valid),
    cat_features=cat_features
)


0:	learn: 90.9486149	test: 90.8587451	best: 90.8587451 (0)	total: 1.36s	remaining: 11m 19s
100:	learn: 14.0694797	test: 14.0953520	best: 14.0953520 (100)	total: 1m 21s	remaining: 5m 21s
200:	learn: 11.2032227	test: 11.2715612	best: 11.2715612 (200)	total: 2m 42s	remaining: 4m 1s
300:	learn: 10.1655985	test: 10.2725717	best: 10.2725717 (300)	total: 4m 14s	remaining: 2m 48s
400:	learn: 9.6253699	test: 9.7365378	best: 9.7365378 (400)	total: 5m 20s	remaining: 1m 19s
499:	learn: 9.3306859	test: 9.4593588	best: 9.4593588 (499)	total: 6m 29s	remaining: 0us

bestTest = 9.459358762
bestIteration = 499



CatBoostRegressor(depth=6, iterations=500, learning_rate=0.05, loss_function='RMSE', task_type='CPU', verbose=100)

In [ ]:
baseline_rmse = 9.459

In [101]:
df['lag_7'] = df.groupby('gitOutlet_ID')['Volume_Liters'].shift(7)

In [102]:
from catboost import CatBoostRegressor

cat_features = list(X_train.select_dtypes(include='object').columns)

model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    verbose=100,
    task_type='CPU',
    thread_count=-1
)
model.fit(
    X_train,
    y_train,
    eval_set=(X_valid, y_valid),
    cat_features=cat_features
)


0:	learn: 90.9486149	test: 90.8587451	best: 90.8587451 (0)	total: 1.61s	remaining: 13m 22s
100:	learn: 14.0694797	test: 14.0953520	best: 14.0953520 (100)	total: 1m 15s	remaining: 4m 56s
200:	learn: 11.2032227	test: 11.2715612	best: 11.2715612 (200)	total: 2m 25s	remaining: 3m 36s
300:	learn: 10.1655985	test: 10.2725717	best: 10.2725717 (300)	total: 3m 31s	remaining: 2m 19s
400:	learn: 9.6253699	test: 9.7365378	best: 9.7365378 (400)	total: 4m 37s	remaining: 1m 8s
499:	learn: 9.3306859	test: 9.4593588	best: 9.4593588 (499)	total: 5m 51s	remaining: 0us

bestTest = 9.459358762
bestIteration = 499



CatBoostRegressor(depth=6, iterations=500, learning_rate=0.05, loss_function='RMSE', task_type='CPU', verbose=100)